# Bonus Lab 03: Exploring Neural Information Processing
## ITAI 4374

**Name:** Win Aung

---

In [1]:
import numpy as np
import matplotlib.pyplot as plt
print('ok lets go')

ok lets go


---
# Part 1: Biological Neuron

## background

neurons communicate using electrical spikes. from the module readings:

- resting potential is like -70mV when nothing is happening
- gotta hit about -55mV threshold to actually fire
- all-or-none means once you cross threshold you get full spike
- refractory period after each spike where it cant fire again

bigger inputs dont make bigger spikes, they make more frequent spikes. thats frequency coding

In [2]:
class BiologicalNeuron:
    def __init__(self):
        self.rest = -70
        self.thresh = -55
        self.peak = 40
        self.refract = 5
    
    def simulate(self, current, steps=200):
        v = self.rest
        volts = []
        spikes = []
        r = 0
        
        for t in range(steps):
            if r > 0:
                r -= 1
                v += (self.rest - v) * 0.2
            else:
                v += current + (self.rest - v) * 0.05
                if v >= self.thresh:
                    v = self.peak
                    spikes.append(t)
                    r = self.refract
            if v > self.rest + 10:
                v = -80
            volts.append(v)
        return volts, spikes

neuron = BiologicalNeuron()
print(f'resting: {neuron.rest}mV, threshold: {neuron.thresh}mV')

resting: -70mV, threshold: -55mV


In [3]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for i, (c, col) in enumerate(zip([0.5, 1.0, 2.0, 4.0], ['green','blue','purple','red'])):
    ax = axes[i//2, i%2]
    v, s = neuron.simulate(c)
    ax.plot(v, color=col, lw=1.5)
    ax.axhline(-55, color='red', ls='--', alpha=0.5)
    ax.set_title(f'Input={c} ({len(s)} spikes)')
    ax.set_ylim(-90, 50)
plt.suptitle('Bio Neuron at Different Input Levels')
plt.tight_layout()
plt.show()

### observations

- 0.5 input barely fires, mostly just sits below threshold
- 1.0 starts getting some spikes
- 2.0 and 4.0 fire way more, more input = more spikes

this shows frequency coding pretty well. spike height stays the same but frequency goes up with stronger input

In [4]:
inputs = np.linspace(0, 5, 25)
rates = [len(neuron.simulate(i, 300)[1]) for i in inputs]

plt.figure(figsize=(8, 5))
plt.plot(inputs, rates, 'b-o', ms=4)
plt.xlabel('Input Current')
plt.ylabel('Spike Count')
plt.title('Firing Rate Curve')
plt.grid(alpha=0.3)
plt.show()

nothing fires below threshold then it ramps up. plateaus at high input because refractory period limits max rate

---
# Part 2: Artificial Neuron

artificial version is simpler. just weighted sum then activation function

In [5]:
class ArtificialNeuron:
    def __init__(self, n_in, act='sigmoid'):
        self.w = np.random.randn(n_in) * 0.5
        self.b = 0
        self.act = act
    
    def forward(self, x):
        z = np.dot(self.w, x) + self.b
        if self.act == 'sigmoid': return 1/(1+np.exp(-z))
        elif self.act == 'relu': return max(0, z)
        elif self.act == 'tanh': return np.tanh(z)
        return z

np.random.seed(42)
art = ArtificialNeuron(3)
print(f'weights: {art.w.round(2)}')
print(f'[0.5, 0.2, 0.8] -> {art.forward([0.5,0.2,0.8]):.3f}')

weights: [ 0.25 -0.07  0.32]
[0.5, 0.2, 0.8] -> 0.549


In [6]:
x = np.linspace(-5, 5, 100)
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for i, (nm, y) in enumerate([('Sigmoid', 1/(1+np.exp(-x))), ('ReLU', np.maximum(0,x)),
                              ('Tanh', np.tanh(x)), ('Leaky ReLU', np.where(x>0,x,0.01*x))]):
    ax = axes[i//2, i%2]
    ax.plot(x, y, lw=2)
    ax.axhline(0, color='gray', lw=0.5)
    ax.axvline(0, color='gray', lw=0.5)
    ax.set_title(nm)
    ax.grid(alpha=0.3)
plt.suptitle('Activation Functions')
plt.tight_layout()
plt.show()

### activation function notes

- sigmoid: squishes to 0-1, has vanishing gradient issue
- relu: just max(0,x), fast but can have dead neurons
- tanh: -1 to 1, zero centered helps training
- leaky relu: small slope for negatives fixes dead neuron problem

---
# Part 3: Networks

In [7]:
class SimpleNetwork:
    def __init__(self, sizes):
        self.W = [np.random.randn(sizes[i], sizes[i+1])*0.5 for i in range(len(sizes)-1)]
    def forward(self, x):
        for w in self.W: x = 1/(1+np.exp(-np.dot(x, w)))
        return x

np.random.seed(42)
net = SimpleNetwork([3,4,2])
pats = [[0,0,0],[1,0,0],[0,1,0],[0,0,1],[1,1,0],[1,0,1],[0,1,1],[1,1,1]]
for p in pats:
    o = net.forward(p)
    print(f'{p} -> [{o[0]:.2f}, {o[1]:.2f}]')

[0, 0, 0] -> [0.62, 0.54]
[1, 0, 0] -> [0.63, 0.55]
[0, 1, 0] -> [0.61, 0.55]
[0, 0, 1] -> [0.63, 0.53]
[1, 1, 0] -> [0.63, 0.56]
[1, 0, 1] -> [0.64, 0.54]
[0, 1, 1] -> [0.63, 0.54]
[1, 1, 1] -> [0.64, 0.55]


In [8]:
outs = np.array([net.forward(p) for p in pats])
plt.figure(figsize=(8, 6))
plt.imshow(outs, aspect='auto', cmap='YlOrRd')
plt.colorbar(label='Activation')
plt.yticks(range(8), [str(p) for p in pats])
plt.xticks([0,1], ['Out1', 'Out2'])
plt.title('Network Output for Each Pattern')
plt.show()

In [9]:
class BioNetwork:
    def __init__(self, n):
        self.neurons = [BiologicalNeuron() for _ in range(n)]
    def run(self, inputs, steps=100):
        return [self.neurons[i].simulate(inputs[i], steps)[1] for i in range(len(inputs))]

np.random.seed(99)
bio = BioNetwork(5)
spks = bio.run([1.5, 0.8, 1.2, 0.5, 1.8], 150)

plt.figure(figsize=(10, 4))
for i, s in enumerate(spks):
    plt.scatter(s, [i+1]*len(s), marker='|', s=100)
plt.xlabel('Time')
plt.ylabel('Neuron')
plt.title('Spike Raster Plot')
plt.ylim(0.5, 5.5)
plt.show()
print('spike counts:', [len(s) for s in spks])

spike counts: [18, 8, 14, 4, 22]


### observations

feedforward net maps inputs to outputs. random weights so not useful yet but shows the info flow

bio network shows different firing rates. neuron 5 had highest input (1.8) so fired most. neuron 4 barely fired with only 0.5 input

---
# Summary

| | Bio | Artificial |
|---|---|---|
| signal | discrete spikes | continuous |
| coding | frequency | activation level |
| threshold | all-or-none | soft |
| timing | has dynamics | instant |
| energy | sparse | dense |

artificial is simpler but misses temporal dynamics. spiking neural networks try to capture some of that, Maass showed they can be more powerful for some tasks. also more energy efficient for neuromorphic hardware